# Circuit from YAML

A kfnetlist `Netlist` can be serialized as YAML. Parse that document with
`yaml.safe_load`, construct the typed netlist with `Netlist.from_dict`, then
bind simulation models explicitly in SAX.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import yaml

import sax
from kfnetlist import Netlist

## MZI

Let's first see how we can define a SAX circuit from YAML:

In [ ]:
netlist = """
instances:
  lft: {kcl: "", component: coupler, settings: {coupling: 0.5}}
  rgt: {kcl: "", component: coupler, settings: {coupling: 0.5}}
  top: {kcl: "", component: straight, settings: {length: 25.0}}
  btm: {kcl: "", component: straight, settings: {length: 15.0}}
ports:
  - {name: in0}
  - {name: in1}
  - {name: out0}
  - {name: out1}
nets:
  - [{instance: lft, port: out0}, {instance: btm, port: in0}]
  - [{instance: btm, port: out0}, {instance: rgt, port: in0}]
  - [{instance: lft, port: out1}, {instance: top, port: in0}]
  - [{instance: top, port: out0}, {instance: rgt, port: in1}]
  - [{name: in0}, {instance: lft, port: in0}]
  - [{name: in1}, {instance: lft, port: in1}]
  - [{name: out0}, {instance: rgt, port: out0}]
  - [{name: out1}, {instance: rgt, port: out1}]
"""
netlist = Netlist.from_dict(yaml.safe_load(netlist))

In [ ]:
netlist.to_dict()

In [ ]:
mzi, _ = sax.circuit(
    netlist,
    models={"coupler": sax.models.coupler_ideal, "straight": sax.models.straight},
)

In [ ]:
wl = jnp.linspace(1.5, 1.6, 1000)
transmission = jnp.abs(mzi(wl=wl)["in0", "out0"]) ** 2

plt.plot(wl * 1e3, transmission)
plt.xlabel("λ [nm]")
plt.ylabel("T")
plt.show()

The YAML document names component factories. SAX uses the model mapping we
supply; we can replace the waveguide model without changing the netlist.

In [ ]:
def waveguide_without_dispersion(wl=1.55, length=25.0, neff=2.34):
    phase = 2 * jnp.pi * neff * length / wl
    sdict = sax.reciprocal({("in0", "out0"): jnp.exp(1j * phase)})
    return sdict

We can regenerate the above circuit again, but this time we specify a models mapping:

In [ ]:
mzi, _ = sax.circuit(
    netlist,
    models={
        "straight": waveguide_without_dispersion,
        "coupler": sax.models.coupler_ideal,
    },
)

`sax.circuit` takes a mapping from component IDs to model functions. The
YAML parser only creates the kfnetlist object; it does not choose models.

In [ ]:
wl = jnp.linspace(1.5, 1.6, 1000)
transmission = jnp.abs(mzi(wl=wl)["in0", "out0"]) ** 2

plt.plot(wl, transmission)
plt.xlabel("Wavelength [nm]")
plt.ylabel("T")
plt.show()